# Novae Benchmarking

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 03.12.2024
- Date of Last Modification: 26.01.2025 (Sebastian Birk; sb75@sanger.ac.uk)

- Run this notebook in the novae conda environment, installable from ```../../envs/novae.yaml```
- This notebook follows the tutorial from https://mics-lab.github.io/novae/tutorials/main_usage/.

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import math
import os

import anndata as ad
import pandas as pd
import novae
import numpy as np
import scanpy as sc
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

### 1.2 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/lustre/scratch126/cellgen/team361/sb75/nichejepa-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'

### 1.3 Define Config

In [ ]:
min_cells_per_niche = 100

### 1.4 Run Notebook Setup

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)

## 2. Zero-shot Niche Identification

### 2.1 chl59-8b_1p (NanoString CosMx Human NSCLC)

#### 2.1.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'chl59-8b_1p'
batches = ['Lung9_Rep1',
           'Lung9_Rep2',
           'Lung5_Rep1',
           'Lung5_Rep2',
           'Lung6',
           'Lung12']
niche_key = 'niche'
batch_key = 'batch'
spot_sizes = [80 * 0.18] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]

            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.18 # conversion rate from https://nanostring-public-share.s3.us-west-2.amazonaws.com/SMI-Compressed/SMI-ReadMe.html

#### 2.1.2 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# Compute zero-shot embeddings
# This is not reproducible
model.compute_representations(adata, zero_shot=True, slide_key=batch_key)

In [ ]:
# Assign niches such that it matches ground truth
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'
model.assign_domains(adata, level=n_domains)

In [ ]:
print(adata.obs[latent_cluster_key].unique().tolist())
print(adata.obs[latent_cluster_key].value_counts())
max_overlap_plot(adata.obs, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
    'D1008',
    'D1000',
    'D1009',
    'D1011',
    'D985',
    'D1003',
    'D998',
    'D989',
    'D1010',
    'D1004',
    'D970',
    'D977',]

In [ ]:
# Plot novae niches
# novae.plot.domains(adata, obs_key=f'novae_domains_{n_domains}')

label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_novae_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=f'novae_domains_{n_domains}',
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.1.3 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'

nmi_scores = []
ari_scores = []
n_niches_list = []

for i in range(8):
    # Load trained novae model
    model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")
    
    model.compute_representations(adata, zero_shot=True, slide_key=batch_key)
    model.assign_domains(adata, level=n_domains)

    n_niches_list.append(adata.obs[latent_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 2.2 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

#### 2.2.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'chr78-11b_1p'
batches = ['CM007',
           'CM008',
           'CM009',
           'CM011',
           'CM013',
           'CM015',
           'CM016',
           'CM017',
           'CM018',
           'CM019',
           'CM020']
niche_key = 'broad_anatomy'
batch_key = 'batch'
spot_sizes = [240 * 0.093] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.093 # conversion rate from Valentina

In [ ]:
# Combine regions
adata.obs[niche_key] = adata.obs[niche_key].apply(lambda x: x.replace('Urogenital Sinus', 'UrogenitalSinus'))

#### 2.2.2 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# Compute zero-shot embeddings
model.compute_representations(adata, zero_shot=True, slide_key=batch_key)

In [ ]:
# Assign niches such that it matches ground truth
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'
model.assign_domains(adata, level=n_domains)

In [ ]:
print(adata.obs[latent_cluster_key].unique().tolist())
print(adata.obs[latent_cluster_key].value_counts())
max_overlap_plot(adata.obs, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
    'D964',
    'D184',
    'D1001',
    'D995',
    'D987',
    'D965',
    'D978',
    'D998',
    'D996',
    'D941',
    'D285',
    'D1002',
    'D962',
    'D893',
    'D985',
    'D1000',
    'D991',
    'D896',
    'D999',
    'D1003']

In [ ]:
# Plot novae niches
# novae.plot.domains(adata, obs_key=f'novae_domains_{n_domains}')

label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_novae_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=f'novae_domains_{n_domains}',
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

In [ ]:
# Store on disk
metrics_df = pd.DataFrame({
    'n_niches': n_niches_list,
    'NMI': nmi_scores,
    'ARI': ari_scores
})
metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

#### 2.2.3 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'

nmi_scores = []
ari_scores = []
n_niches_list = []

for i in range(8):
    # Load trained novae model
    model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")
    
    model.compute_representations(adata, zero_shot=True, slide_key=batch_key)
    model.assign_domains(adata, level=n_domains)

    n_niches_list.append(adata.obs[latent_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 2.3 xhs49-1b_1p (Xenium Human Melanoma)

#### 2.3.1 Load Data

In [ ]:
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
spot_size = 20

In [ ]:
adata = sc.read_h5ad(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/{dataset_name}/Xenium_Prime_Human_Skin_FFPE_outs.h5ad')

# Filter cells without niche labels
adata = adata[~adata.obs['niche_name'].isna()]

In [ ]:
adata.obs['niche_name'].value_counts()

#### 2.3.2 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Load trained novae model
model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# Compute zero-shot embeddings
model.compute_representations(adata, zero_shot=True)

In [ ]:
# Assign niches such that it matches ground truth
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'
model.assign_domains(adata, level=n_domains)

In [ ]:
print(adata.obs[latent_cluster_key].unique().tolist())
print(adata.obs[latent_cluster_key].value_counts())
max_overlap_plot(adata.obs, niche_key, latent_cluster_key)

In [ ]:
inferred_niches_ordered = [
     'D1011',
     'D1014',
     'D1012',
     'D993',
     'D1016',
     'D1015',
     'D1008']

In [ ]:
# Plot novae niches
# novae.plot.domains(adata, obs_key=f'novae_domains_{n_domains}')

label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

file_name = f'{dataset_name}_cellplm_niches.svg'
sc.pl.spatial(adata,
              color=latent_cluster_key,
              palette=pred_palette,
              spot_size=spot_size,
              save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.3.3 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
n_domains = (adata.obs[niche_key].value_counts() > min_cells_per_niche).sum() # ground truth number with > 100 cells
latent_cluster_key = f'novae_domains_{n_domains}'

nmi_scores = []
ari_scores = []
n_niches_list = []

for i in range(8):
    # Load trained novae model
    model = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")
    
    model.compute_representations(adata, zero_shot=True)
    model.assign_domains(adata, level=n_domains)

    n_niches_list.append(adata.obs[latent_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{artifacts_folder_path}/benchmarking/results/{dataset_name}_novae_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")